# Semana 2 · Experimento del estudiante: tokens, atención y decoding

Este notebook acompaña la sesión. En cada sección hay una **predicción** que se escribe antes de ejecutar y un **TODO** para experimentar. Corre completo en Colab con CPU.

Reglas:

1. Escribir la predicción antes de correr la celda que la verifica.
2. Al final, responder las preguntas de cierre en la celda de texto.

Entregable: este notebook ejecutado, con predicciones, resultados y respuestas.

In [ ]:
# Instalación (en Colab tarda un minuto; si las librerías ya están instaladas no hace nada)
import subprocess, sys
_ = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "torch", "matplotlib"],
                   capture_output=True)

In [ ]:
import os, time, warnings
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(0)
np.set_printoptions(precision=3, suppress=True)

tok = AutoTokenizer.from_pretrained("gpt2")
# attn_implementation="eager" es necesario para obtener los pesos de atención con output_attentions=True
model = AutoModelForCausalLM.from_pretrained("gpt2", dtype=torch.float32, attn_implementation="eager")
model.eval()

cfg = model.config
print(f"vocabulary:        {cfg.vocab_size:,} tokens")
print(f"context window:    {cfg.n_positions:,} posiciones")
print(f"d_model:           {cfg.n_embd} (dimensión de cada vector)")
print(f"bloques:           {cfg.n_layer}   cabezas por bloque: {cfg.n_head}")
print(f"parámetros:        {sum(p.numel() for p in model.parameters())/1e6:.0f} M")

In [ ]:
def limpiar(t):
    """Los tokens de GPT-2 marcan el espacio previo con 'Ġ'. Se cambia por '␣' para leerlos."""
    return t.replace("Ġ", "␣").replace("Ċ", "\\n")

def mostrar_tokens(texto):
    ids = tok(texto).input_ids
    toks = [limpiar(t) for t in tok.convert_ids_to_tokens(ids)]
    print(f"{texto!r}")
    print(f"  palabras: {len(texto.split())}   tokens: {len(ids)}   tokens/palabra: {len(ids)/len(texto.split()):.2f}")
    print(f"  tokens:   {toks}")
    print(f"  ids:      {ids}")
    return ids

## 1. Tokens

**Predicción 1.** ¿Cuántos tokens tiene la frase de abajo en GPT-2? Escribe tu número en `MI_PREDICCION` antes de ejecutar.

In [ ]:
frase = "Arquitectura de software para sistemas inteligentes"
MI_PREDICCION = None   # TODO: escribe un entero antes de ejecutar

ids = mostrar_tokens(frase)
print(f"\npredicción: {MI_PREDICCION}   real: {len(ids)}")

In [ ]:
# TODO: escribe una frase tuya en español y la misma idea en inglés. Compara tokens por palabra.
mi_es = "TODO: frase en español"
mi_en = "TODO: same sentence in English"
_ = mostrar_tokens(mi_es)
_ = mostrar_tokens(mi_en)

**Pregunta 1.** ¿Por qué el español necesita más tokens que el inglés con este tokenizer? ¿Qué consecuencia tiene para el costo y la ventana de contexto de una aplicación en español?

_Respuesta:_ (escribe aquí)

## 2. Embeddings

Cada token ID selecciona una fila de una matriz aprendida. Tokens que aparecen en contextos parecidos quedan con vectores parecidos.

In [ ]:
wte = model.transformer.wte.weight
print("matriz de embeddings:", tuple(wte.shape))

def similitud(a, b):
    va, vb = wte[tok(a).input_ids[0]], wte[tok(b).input_ids[0]]
    return torch.cosine_similarity(va, vb, dim=0).item()

# TODO: ordena estos pares de mayor a menor similitud ANTES de ejecutar, en MI_ORDEN (lista de índices 0..3).
pares = [(" cat", " dog"), (" cat", " car"), (" king", " queen"), (" king", " banana")]
MI_ORDEN = None   # por ejemplo [0, 2, 1, 3]

for i, (a, b) in enumerate(pares):
    print(f"{i}: {a!r:8} vs {b!r:9} -> {similitud(a, b):.3f}")
print("mi orden:", MI_ORDEN)

## 3. Self-attention

### 3.1 A mano

La celda de abajo reproduce el ejercicio en papel. Antes de ejecutarla, calcula a mano la fila de 'gato' (pesos con máscara causal) y escríbela en `MI_FILA_GATO`.

In [ ]:
MI_FILA_GATO = None   # TODO: por ejemplo [0.5, 0.5, 0.0]

# Tres tokens con embeddings de dimensión 2. En GPT-2 la dimensión es 768 y las matrices W se aprenden.
tokens = ["el", "gato", "duerme"]
X = np.array([[1.0, 0.0],
              [0.0, 1.0],
              [1.0, 1.0]])
W_Q = np.array([[1.0, 0.0], [0.0, 1.0]])
W_K = np.array([[0.0, 1.0], [1.0, 0.0]])   # intercambia coordenadas: la key de un token difiere de su query
W_V = np.array([[1.0, 0.0], [0.0, 1.0]])

Q, K, V = X @ W_Q, X @ W_K, X @ W_V
d_k = Q.shape[1]
scores = Q @ K.T / np.sqrt(d_k)
mask = np.triu(np.ones_like(scores, dtype=bool), k=1)       # True donde j > i (futuro)
scores_masked = np.where(mask, -np.inf, scores)

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

A = softmax(scores_masked)
salida = A @ V
print("scores / sqrt(d_k):\n", scores)
print("\ncon máscara causal:\n", scores_masked)
print("\npesos de atención:\n", A)
print("\nsalida = A @ V:\n", salida)
for i, t in enumerate(tokens):
    print(f"  '{t}' mira a: " + ", ".join(f"{tokens[j]}={A[i, j]:.2f}" for j in range(i + 1)))
print("\nmi fila de 'gato':", MI_FILA_GATO, "  real:", A[1])

**Pregunta 2.** Cambia `W_K` por la identidad (igual que `W_Q`) y vuelve a ejecutar. ¿Qué cambia en la fila de 'gato' y por qué?

_Respuesta:_ (escribe aquí)

### 3.2 Atención real en GPT-2

In [ ]:
def mapa_atencion(texto, capa, cabeza, ax=None):
    ids = tok(texto, return_tensors="pt").input_ids
    labels = [limpiar(t) for t in tok.convert_ids_to_tokens(ids[0])]
    with torch.no_grad():
        out = model(ids, output_attentions=True)
    att = out.attentions[capa][0, cabeza].numpy()      # (tokens, tokens)
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(att, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right"); ax.set_yticklabels(labels)
    ax.set_xlabel("Key (a quién se mira)"); ax.set_ylabel("Query (quién mira)")
    ax.set_title(f"capa {capa}, cabeza {cabeza}")
    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, hatch="///", fill=False, edgecolor="gray", lw=0))
    return att, labels

In [ ]:
frase = "The cat sat on the mat because it was tired"
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
att3, labels = mapa_atencion(frase, capa=4, cabeza=3, ax=axes[0])
att11, _ = mapa_atencion(frase, capa=4, cabeza=11, ax=axes[1])
plt.tight_layout(); plt.show()

# TODO: explora otra capa y otra cabeza (0 a 11 cada una). ¿Encuentras alguna con un patrón que puedas describir?
MI_CAPA, MI_CABEZA = 0, 0
att_mia, _ = mapa_atencion(frase, capa=MI_CAPA, cabeza=MI_CABEZA)
plt.show()

**Pregunta 3.** ¿Por qué el triángulo superior de cada mapa está bloqueado? ¿Qué aprendería el modelo si no lo estuviera durante el entrenamiento?

_Respuesta:_ (escribe aquí)

## 4. Logits y decoding

**Predicción 2.** ¿Cuál es el token más probable después de `"The cat sat on the"`? Escríbelo en `MI_TOKEN`.

In [ ]:
MI_TOKEN = None   # TODO: por ejemplo " floor"

ids = tok("The cat sat on the", return_tensors="pt").input_ids
with torch.no_grad():
    ultimo = model(ids).logits[0, -1]
probs = torch.softmax(ultimo, dim=-1)
top = torch.topk(probs, 10)
for p, i in zip(top.values, top.indices):
    print(f"  {tok.decode([i])!r:12} prob={p:.3f}")
print("\nmi predicción:", MI_TOKEN)

In [ ]:
# La misma distribución a distintas temperaturas. TODO: cambia los tres valores de T y observa.
MIS_TEMPERATURAS = [0.3, 1.0, 2.0]
cand = top.indices
fig, axes = plt.subplots(1, len(MIS_TEMPERATURAS), figsize=(13, 3.5), sharey=True)
for ax, T in zip(axes, MIS_TEMPERATURAS):
    p = torch.softmax(ultimo / T, dim=-1)[cand].numpy()
    ax.bar(range(10), p)
    ax.set_xticks(range(10)); ax.set_xticklabels([limpiar(tok.decode([i])) for i in cand], rotation=45, ha="right")
    ax.set_title(f"temperature = {T}")
plt.tight_layout(); plt.show()

In [ ]:
def generar(prefijo, n=3, max_new_tokens=15, **kw):
    ids = tok(prefijo, return_tensors="pt").input_ids
    return [tok.decode(model.generate(ids, max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id, **kw)[0, ids.shape[1]:]) for _ in range(n)]

prefijo = "The software architect opened the report and"

# TODO: completa la tabla. Para cada configuración, predice cuántas salidas DISTINTAS habrá en 3 corridas (1, 2 o 3).
configs = {
    "greedy":          (dict(do_sample=False),                 None),   # (configuración, mi predicción)
    "temperature=0.2": (dict(do_sample=True, temperature=0.2), None),
    "temperature=1.5": (dict(do_sample=True, temperature=1.5), None),
    "top-p=0.9":       (dict(do_sample=True, top_p=0.9, top_k=0), None),
}
for nombre, (cfg, pred) in configs.items():
    salidas = generar(prefijo, **cfg)
    print(f"\n=== {nombre}: predije {pred}, distintas = {len(set(salidas))} ===")
    for x in salidas:
        print("  ", repr(x))

**Pregunta 4.** Cuando subes la temperatura, ¿qué cambia en los logits? ¿Y en las probabilidades? Explica en dos líneas.

_Respuesta:_ (escribe aquí)

**Pregunta 5.** Para clasificar tickets de soporte en cinco categorías, ¿qué estrategia de decoding usarías y por qué? ¿Y para proponer nombres de un producto?

_Respuesta:_ (escribe aquí)

## 5. Ventana de contexto

In [ ]:
# Predicción 3: ¿qué pasa si se envían 1,100 tokens a un modelo con ventana de 1,024?
MI_PREDICCION_VENTANA = None   # TODO: "error", "trunca", "funciona"

# Desbordamiento del context window: GPT-2 conoce 1,024 posiciones.
texto_largo = "token " * 1100
ids = tok(texto_largo, return_tensors="pt").input_ids
print("tokens de entrada:", ids.shape[1], "| ventana:", model.config.n_positions)
try:
    with torch.no_grad():
        model(ids)
    print("sin error (¿?)")
except Exception as e:
    print("ERROR:", type(e).__name__, "->", str(e)[:160], "...")
print("\nmi predicción:", MI_PREDICCION_VENTANA)

## 6. Cierre

Responde en esta celda:

1. Explica con tus palabras por qué el modelo de la semana 1 dio respuestas distintas cada vez.
2. Explica por qué inventó una fecha sin "darse cuenta".
3. Para la mesa de soporte del curso: ¿cuántos tokens estimas que ocupa este ticket? "Me cobraron dos veces la mensualidad de julio. Necesito el reembolso del cargo duplicado." Escribe tu estimación, tokenízalo y anota el número real.

_Respuestas:_ (escribe aquí)